# Lecture 0. Intro to Motion Planning for Manipulators

## 1. 什么是运动规划？

从根本上说，机械臂运动规划的核心问题是寻找一个有效且连续的配置序列（即路径或轨迹），该序列能够将机器人从一个起始状态移动到一个期望的目标状态，同时不违反任何约束条件，其中最关键的约束是避免与自身或环境中的障碍物发生碰撞 。这个基本问题可以进一步分解为两个子问题：路径规划（Path Planning）和轨迹规划（Trajectory Planning）。路径规划专注于寻找一条纯粹的几何路径，而轨迹规划则为这条路径赋予时间规律，即定义其速度和加速度曲线 。   

<figure style="text-align: center; margin: 20px 0;">
    <video width="720" controls style="display: block; margin: 0 auto;" autoplay muted loop>
        <source src="https://daoming-chen.github.io/MP_lecture_assets/lec0/obs_avoid_example.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            Son D, Jung H, Kim B. NeuralSVCD for Efficient Swept Volume Collision Detection[J]. <br>
            arXiv preprint arXiv:2509.00499, 2025.
    </figcaption>
</figure>

在现代机器人技术中自主规划运动的能力，是将机器人从一个简单的遥控设备转变为智能体的关键。这项能力是现代机器人技术的基石，它催生了众多深刻影响生产力和社会的应用 。尤其是操作 (Manipulation) 方面。   

工业自动化：在制造业中，运动规划对于焊接、复杂零件组装和码垛等精密任务至关重要。在这些结构化环境中，机器人必须执行精确且可重复的运动 。 

<figure style="text-align: center; margin: 20px 0;">
    <video width="720" controls style="display: block; margin: 0 auto;" autoplay muted loop>
        <source src="https://daoming-chen.github.io/MP_lecture_assets/lec0/tesla_factory.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            特斯拉上海工厂宣传片
    </figcaption>
</figure>

  

医疗健康：在医疗领域，机械臂辅助进行需要高度精确性的外科手术，帮助患者康复，并执行实验室自动化任务 。

<figure style="text-align: center; margin: 20px 0;">
    <video width="720" controls style="display: block; margin: 0 auto;" autoplay muted loop>
        <source src="https://daoming-chen.github.io/MP_lecture_assets/lec0/davinci_robot.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            达芬奇手术机器人的宣传片
    </figcaption>
</figure>

服务与协作机器人：随着机器人逐渐进入以人为中心的环境，挑战也随之加剧。协作机器人（Cobots）必须能够安全、可预测地在人类周围规划运动，并适应家庭、仓库和医院等动态、非结构化的环境 。   

<figure style="text-align: center; margin: 20px 0;">
    <video width="720" controls style="display: block; margin: 0 auto;" autoplay muted loop>
        <source src="https://daoming-chen.github.io/MP_lecture_assets/lec0/figure3.mp4" type="video/mp4">
    </video>
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
            Figure 03 服务机器人的宣传片
    </figcaption>
</figure>


## 2. 什么是“好的”规划？

运动规划研究的最初焦点仅仅是找到任何可行的解决方案，这一概念被称为“完备性”（Completeness）。比如最经典的 Dijkstra 算法

<div style="text-align: center; margin: 20px 0;">
    <img src="https://daoming-chen.github.io/MP_lecture_assets/lec0/Dijkstra_viz.gif" width="600" style="display: block; margin: 0 auto;">
    <p style="margin-top: 10px; font-size: 16px; line-height: 1.5;">
        Dijkstra 算法可视化 <br>
    </p>
</div>

> 可以在 <a href="https://claude.ai/public/artifacts/a1f3e3ba-d0c9-4d82-ac01-639becb90bc9">Dijkstra 算法可视化</a> 自行体验与上图类似的效果

然而，现代应用的要求远不止于此。一个“好”的规划需要通过多个标准来评估：   

- 可行性（Feasibility）：规划出的路径必须是无碰撞的，并且要遵守机器人的运动学和动力学约束 。   
- 最优性（Optimality）：规划应在给定成本函数下达到“最优”。这可能意味着最小化路径长度、执行时间或能耗 。   
- 平滑性与可预测性（Smoothness & Predictability）：为了安全可靠地执行，轨迹应该是平滑的（即具有较低的加加速度和加速度）。在工业和协作场景中，可预测性至关重要；相似的规划查询应该产生相似的路径 。

<div style="text-align: center; margin: 20px 0;">
    <img src="https://github.com/quimortiz/dynoplan/assets/32126190/b14905b7-8a8b-435e-be6e-11dfc49f909a" width="600" style="display: block; margin: 0 auto;">
    <p style="margin-top: 10px; font-size: 16px; line-height: 1.5;">
        iDb-A* Optimal Trajectory Planning <br>
        de Haro J O, Hönig W, Hartmann V N, et al. <br>
        iDb-A*: Iterative search and optimization for optimal kinodynamic motion planning[J]. <br>IEEE Trans. Robotics, 2025.
    </p>
</div>

这种从可行性到多目标最优性的演变，清晰地反映了机器人技术领域的成熟过程。

早期的研究工作主要关注一个二元问题：“是否存在一条路径？”能够保证在路径存在时找到它的算法被称为“完备的”，但对于复杂机器人来说，这些算法的计算成本往往高得令人望而却步。这一计算瓶颈催生了基于采样的规划方法，它们牺牲了绝对的完备性，以换取概率完备性和实践中的高效率，但其初始解往往不平滑且非最优。

这些初始解的质量不佳，反过来又催生了对“更好”路径的需求，从而引入了最优性的概念（例如最短路径），并推动了像 RRT* 这样的算法的发展。

<div style="text-align: center; margin: 20px 0;">
    <img src="https://daoming-chen.github.io/MP_lecture_assets/lec0/rrt_star.png" width="600" style="display: block; margin: 0 auto;">
    <p style="margin-top: 10px; font-size: 16px; line-height: 1.5;">
        RRT -> RRT* <br>
        Karaman S, Frazzoli E. Incremental sampling-based algorithms for optimal motion planning[J]. <br>
        Robotics Science and Systems VI, 2010, 104(2): 267-274.
    </p>
</div>

随着机器人变得更加动态，并以更复杂的方式与世界互动，单一的路径长度度量已不再足够。电机扭矩、能量消耗以及流畅运动所需的平滑度等因素变得至关重要。这最终导致了轨迹优化框架（如 CHOMP、STOMP、TrajOpt）的兴起，这些框架通过定义丰富的、包含多个项的成本函数来构建问题。

<div style="text-align: center; margin: 20px 0;">
    <img src="https://daoming-chen.github.io/MP_lecture_assets/lec0/trajopt.gif" width="600" style="display: block; margin: 0 auto;">
    <p style="margin-top: 10px; font-size: 16px; line-height: 1.5;">
        TrajOpt <br>
        Schulman J, Duan Y, Ho J, et al. Motion planning with sequential convex optimization and convex collision checking[J]. <br>
        The International Journal of Robotics Research, 2014, 33(9): 1251-1270.
    </p>
</div>

这代表了现代运动规划的观点：它不再仅仅是一个搜索问题，而是一个整体的优化问题。这一趋势表明，随着机器人越来越多地在现实世界中承担复杂任务，运动的质量变得与运动的存在本身同等重要。这种转变要求规划器能够处理复杂的、通常是不可微的成本函数和约束。